# CodeReview-Env — GRPO Training (Colab T4)

Train a code review agent using Unsloth + HF TRL GRPOTrainer.
Uses the live CodeReview-Env on HuggingFace Spaces for reward scoring.

In [ ]:
# Cell 1: Install dependencies
%pip install unsloth trl transformers datasets requests pydantic matplotlib -q
%pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
print("Installation complete")

In [ ]:
# Cell 2: Configuration
import os

os.environ["ENV_URL"] = "https://dharaneswarreddy-codereview-env.hf.space"
os.environ["MOONSHOT_API_KEY"] = "your_moonshot_key"  # from https://platform.kimi.ai/console/api-keys
os.environ["HF_TOKEN"] = "your_hf_token_here"
os.environ["TRAIN_MODEL"] = "Qwen/Qwen2.5-1.5B-Instruct"
os.environ["SAVE_REPO"] = "DharaneswarReddy/codereview-agent"

print("Config set:")
print(f"  ENV_URL: {os.environ['ENV_URL']}")
print(f"  TRAIN_MODEL: {os.environ['TRAIN_MODEL']}")
print(f"  SAVE_REPO: {os.environ['SAVE_REPO']}")

In [ ]:
# Cell 3: Clone repo and run training
!git clone https://github.com/GojoV339/MetaXScaler_Hackathon.git codereview-env 2>/dev/null || true
%cd codereview-env

# Run training
exec(open('training/train_grpo.py').read())

In [ ]:
# Cell 4: Show reward curve
from IPython.display import Image
Image("reward_curve.png")

In [ ]:
# Cell 5: Test trained model
import requests, json

ENV_URL = os.environ["ENV_URL"]
obs = requests.post(f"{ENV_URL}/reset", json={"task_level": 1}).json()
print(f"Snippet: {obs['snippet_id']}")
print(f"Code:\n{obs['code']}\n")

# Run trained model
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

prompt = make_prompt(obs, tokenizer, level=1)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
import torch
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.2, do_sample=True)
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"Model response:\n{response}")

action = parse_action(response)
result = requests.post(f"{ENV_URL}/step", json=action).json()
print(f"\nScore: {result['reward']['score']:.3f}")
print(f"Feedback: {result['reward']['feedback']}")

In [ ]:
# Cell 6: Compare with Kimi K2.6 baseline (optional)
from openai import OpenAI

kimi_client = OpenAI(
    base_url="https://api.moonshot.ai/v1",
    api_key=os.environ["MOONSHOT_API_KEY"],
)

test_obs = requests.post(f"{ENV_URL}/reset", json={"task_level": 1}).json()
kimi_resp = kimi_client.chat.completions.create(
    model="kimi-k2.6",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPTS[1]},
        {"role": "user", "content": f"Review this code:\n```python\n{test_obs['code']}\n```\nReturn ONLY the JSON."}
    ],
    max_tokens=200,
    temperature=0.6,
    extra_body={"thinking": {"type": "disabled"}},
)
print(f"Kimi K2.6 response: {kimi_resp.choices[0].message.content}")